---
SECTION 1 – Python Fundamentals & Async (10 minutes)
---
---




---
**Q1. (MCQ)**

What happens when an async def FastAPI endpoint calls a blocking function (e.g. time.sleep())?

---

A. FastAPI automatically converts it to non-blocking

B. The event loop is blocked, affecting concurrent requests

C. It runs in a separate thread automatically

D. Only that request is delayed, others are unaffected



**Answer: B.**

Regular sleep would block the entire event loop. For non blocking sleep that only affect that specific task we need to use asyncio.sleep()


---
**Q2. (Short Answer)**
Explain why FastAPI performs well compared to Flask under high concurrency.

---

(2–3 sentences max)



**FastAPI works as an abstraction layer on top of an async network engine like uvicorn.**
**That engine runs multiple workers, each with it's own event loop, scheduling and calling async fastapi and httpx methods.**
**Flask is built on a synchronous engine so each worker can only handle one incoming request at a time.**
**It can still create an event loop after getting a request but it would still be able to handle only one request at a time so there is no io bound concurrency.**

---
Q3. (Code Reading)
What is the output?

---
```python
async def foo():
    return 1

def bar():
    return foo()

result = bar()
print(result)
A. 1
B. <coroutine object foo>
C. Raises RuntimeError
D. None
```

**Answer: B**

foo is an async function e.g. we can await other async functions in it's body AND it returns a coroutine.
bar is a synchronous function so the code inside it's body is blocking, it's executed while bar is called and we can't yield or await things inside its body.
bar returns a courutine so outside of bar, it's as if it was async. 

**Therefore, just running bar or foo outside of an event loop would just print the __str__ of the coroutine object.**

---
SECTION 2 – FastAPI Core (15 minutes)
---
---

---
Q4. (MCQ)
Which FastAPI feature enables automatic request validation?

---

A. Marshmallow

B. SQLAlchemy

C. Pydantic

D. Starlette


**Answer: C**

Marshmallow - serialisation/desiarlisation library that has schema validation but it's less type dependent and not built in fastapi. Pydantic mostly replaces it.

SQLAlchemy - an ORM library that abstracts SQL operations and allows running native SQL.

Pydantic - built in FastAPI, a schema validation library, strongly typed.

Starlette - the underlying library that handles the async web handling, routing and connection to the web engine like uvicorn.
FastAPI is built on top of Starlette and adds pydantic validation, dependency injection and other high level functionality.




---
Q5. (Code Completion)
Complete this endpoint so that:

---

email is required

age must be ≥ 18

response returns validated data

```python
from fastapi import FastAPI
from pydantic import BaseModel, EmailStr, Field

app = FastAPI()

class UserCreate(BaseModel):
    email: EmailStr
    age: int = Field(..., ge=18)

@app.post("/users")
def create_user(user: UserCreate):
    return ____
```


In [ ]:
# Answer: We just neet to return user, pydantic does the validation automatically
# and fastapi creates a response and handles errors automatically.

from fastapi import FastAPI
from pydantic import BaseModel, EmailStr, Field

app = FastAPI()


class UserCreate(BaseModel):
    email: EmailStr
    age: int = Field(..., ge=18)


@app.post("/users")
def create_user(user: UserCreate):
    return user


---
Q6. (Conceptual)
Explain FastAPI dependency injection (Depends) and give one concrete use case.

---


**Answer:**

FastAPI's dependency injection is done via a class called Depends that holds a factory/provider function that returns the dependency.
When calling Depends(factory_of_a_thing) in a handler function, when the route is invoked and the handler funciton is called,
fastAPI calls Depends.dependency(factory_of_a_thing) -> thing that returns the dependency.
We can put cleanup and life cycle logic in the factory function in a try - yield - except - finally block and fastapi would take care of running the cleanup logic in the finally block.


Example:
```python
from fastapi import Depends
from my_logging import logger_singleton
from db_lib import Connection
from typing import Generator

def db_con(connection_string: str,
logger: Logger = Depends(logger_singleton)
) -> Generator[Connection, None, None]:
    try:
        con: Connection = db_lib.connect(connection_string)
        yield con
    
    except Exception as e:
        logger.error("Failed to connect to db: %s", e)
        raise e

    finally:
        if con and not con.closed():
            con.close()
            logger.info("Closed connection.")


```

---
Q7. (MCQ)
Where should database session cleanup ideally occur?

---

A. Inside the route handler

B. Inside middleware only

C. Inside a dependency using yield

D. In __del__ of the session object



**Answer: C**

Ideally, the connection/session cleanup would be handled inside a finally block in a dependency holding the connection factory, after a yield statement.
The yield statement returns the active connection and yields the rest of the execution. When the response is sent, fastAPI calls next() on the 
factory funciton of the db connection and the finally block is executed.

---
SECTION 3 – Databases & Data Flow (10 minutes)
---
---



---
Q8. (Short Answer)
Why is async database access important in FastAPI applications?

---


**Answer:**

Async access is important in fastapi applications because the strength of fastapi is handling massive async traffic.
Without async connections to the db, one connection would block our worker and prevent it from handling multiple requests.
This would negate the whole point of using fastAPI.

Therefore, a connection pool of a few async db connections would not block a worker since it would be able to work with multiple connections from the pool for multiple requests.This would allow us to work with high traffic load without putting the load on the db and without blocking our workers.


---
Q9. (Scenario)
You need to build a high-throughput read API (thousands of requests/sec).
Which choices improve scalability? (Select ALL)

---

A. Async endpoints

B. Connection pooling

C. Blocking ORM calls

D. Caching (Redis)

E. Global DB session object


**Answer: A, B, D**


---
Q10. (Design)
How would you implement pagination in a REST API?
(List parameters + behavior)

---


I would add query parameters page_size and page_number and create an offset from them:
offset: int = page_number * page_size and then use the offset as an offset in the database query that pulls the records.
```SQL
select * from records
offset $1
limit $2;
offset, page_size
;
```

```python
import pandas
from fastapi import FastAPI, Depends

from .domain.models import Record
from .infrastructure.db.postgres import DB, get_connection

app = FastAPI()

app.get('/get_records')
def paginate(page_size: int=10, page_number: int=1, db: DB = Depends(get_connection)) -> list[Record]:
    return db.get_records(offset=page_size * page_number, limit=page_size)

```


---
SECTION 4 – Authentication & Security (15 minutes)
---
---



---
Q11. (MCQ)
In FastAPI, OAuth2 with JWT typically uses which dependency?

---

A. HTTPBearer

B. OAuth2PasswordBearer

C. Depends(JWT)

D. SecurityScopes


**Answer: B**

OAuth2PasswordBearer is a class that its underlying __call__ method extracts the JWT token from the Authorisation header of the request.


---
Q12. (Short Answer)
What security risks exist if JWT tokens are stored in localStorage?

---


localStorage is a part of the browser that contains data accessible by JavaScript code, if JWTs are stored there they are exposed to cross side scripting attacks
and could be stolen and grant access to malicious code.

To fix this, tokens should be stored in cookies (the browser stores them in the cookie store) with the HTTP-only flag set to true so that the flag tells the browser to block the access to JS code for the cookie.


---
Q13. (Code Reasoning)
What is wrong with this code?

---

```python
@app.get("/secure")
def secure(token: str):
    return {"token": token}
```

**Answer:** 

The token is passed as a query parameter instead of a secure way (HTTP-only, Secure cookie). That makes the token exposed to external scripts and data leakage. Any script that tracks the url (client side or third party) can see and copy the token making the app vulnerable to unauthorized login.


---
Q14. (Design)
Describe a role-based access control (RBAC) approach in FastAPI.

---


**Answer:**

Role based access control means that users (and their tokens) are assigned with roles, each role is a set of permissions. Therefore, RBAC makes the app more secure as only tokens with specific permissions are authorised to access to some of the resources.


---
SECTION 5 – Docker, Cloud & CI/CD (15 minutes)
---
---


---
Q15. (MCQ)
Which command typically runs a FastAPI app in production?

---

A. python main.py

B. flask run

C. uvicorn main:app --workers 4

D. fastapi start


**Answer: C**


---
Q16. (Dockerfile Understanding)
Why is this line important?

---

ENV PYTHONUNBUFFERED=1


**Answer:**

This environment variable tells docker to unbuffer the python output and pass it to stdout and stderr
so the output can be seen from the docker logs.


---
Q17. (Scenario)
You deploy to Azure. Where should secrets (API keys, DB passwords) live?

---

A. Dockerfile

B. GitHub repository

C. Azure Key Vault / environment variables

D. Inside source code


**Answer: C**




---
Q18. (CI/CD)
List three pipeline stages you would enforce before production deployment.

---


**Answer:**

1. Pull code / image + build.
2. Tests (linting, functional, regression).
3. Merge / deploy into production environment.

---
SECTION 6 – AI/ML API Integration (10 minutes)
---
---

---
Q19. (Short Answer)
What are two risks when calling external AI APIs (OpenAI, Hugging Face) from a backend?

---


**Answer:**

1. Data Leakage: internal data (source code, strategies etc) is passed to the model supplier which is third party. This is a risk to privacy.
2. Maintainability: dependency of a specific model could create a vendor lock and updates in the third party's API can break the connection.

---
Q20. (Design)
Describe a clean architecture for integrating AI inference into a FastAPI backend.

---

**Answer:**

Assuming we're integrating inference to an existing clean architecture app.
External service: Set up liteLLM as a reverse proxy to connect to model providers.
1. Infrastructure layer:
    Create a class for AI operations (AIProvider) with a method run_inference.
2. Presentation layer:
    Set up a dependencies module, in this module create a FastAPI fetch function or dependency_injector singleton that initialises and/or
    fetches the AIProvider class.
    In addition, create a provider for a token validation singleton that accepts a JWT and validates its signature.
3. Domain layer:
    Set up a models module that has data classes that inherit from pydantic's BaseModel class for:
    - The request to the AI model, including a hard coded or user provided prompt.
    - The response of the model with the inferred data.
4. Application layer:
    Set up a FastAPI.post endpoint with an OAuthHTTPPasswordProvider dependency and a token validation dependency.
    The endpoint's path function should return the response model.
